# Step 3 - BERT 임베딩 + MLP 분류기

- `extract_embeddings.ipynb` 실행 후 진행 (npy 로드만, BERT 재추출 X)
- MLP: 768 -> 256 -> 64 -> 10, ReLU, Dropout 0.3
- 클래스 불균형 -> class weight 적용, **주 지표 F1-macro**
- 목표: baseline TF-IDF+SVM **F1-macro 0.6347** 돌파
- test/val 예측확률을 저장 (Late Fusion에서 재사용)

In [ ]:
import os, time, numpy as np, pandas as pd

# 스모크 테스트: True면 클래스별 소량 샘플만 사용 (파이프라인 점검용)
SMOKE_TEST = False
N_SMOKE = 1000

def find_root(start='.'):
    p = os.path.abspath(start)
    while p != os.path.dirname(p):
        if os.path.exists(os.path.join(p, 'processed_data')):
            return p
        p = os.path.dirname(p)
    return os.path.abspath(start)

ROOT = find_root()
DATA_DIR = os.path.join(ROOT, 'processed_data')
ART_DIR = os.path.join(ROOT, 'artifacts')
os.makedirs(ART_DIR, exist_ok=True)
print('ROOT     :', ROOT)
print('DATA_DIR :', DATA_DIR)
print('ART_DIR  :', ART_DIR)

In [ ]:
# 라벨 순서 (label_mapping.csv 기준: 인덱스 = label 값)
LABEL_NAMES = ['가사', '개인정보/ICT', '근로자', '금융조세', '기업',
               '민사', '특허/저작권', '행정', '형사A(생활형)', '형사B(일반형)']
N_CLASS = 10

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

In [ ]:
def load_emb(split):
    X = np.load(os.path.join(ART_DIR, f'bert_{split}_X.npy'))
    y = np.load(os.path.join(ART_DIR, f'bert_{split}_y.npy'))
    return X.astype('float32'), y.astype('int64')

Xtr, ytr = load_emb('train')
Xval, yval = load_emb('val')
Xte, yte = load_emb('test')
print('train', Xtr.shape, 'val', Xval.shape, 'test', Xte.shape)

In [ ]:
counts = np.bincount(ytr, minlength=N_CLASS)
w = counts.sum() / (N_CLASS * np.maximum(counts, 1))
class_weight = torch.tensor(w, dtype=torch.float32).to(device)
print('class counts:', counts.tolist())

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim=768, n_cls=N_CLASS, p=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256), nn.ReLU(), nn.Dropout(p),
            nn.Linear(256, 64), nn.ReLU(), nn.Dropout(p),
            nn.Linear(64, n_cls))
    def forward(self, x):
        return self.net(x)

In [ ]:
torch.manual_seed(42)
mlp = MLP().to(device)
crit = nn.CrossEntropyLoss(weight=class_weight)
opt = torch.optim.Adam(mlp.parameters(), lr=1e-3)
tr_loader = DataLoader(TensorDataset(torch.tensor(Xtr), torch.tensor(ytr)),
                       batch_size=128, shuffle=True)
Xval_t = torch.tensor(Xval).to(device)

best_f1, best_state, patience, bad = -1, None, 7, 0
for epoch in range(60):
    mlp.train()
    for xb, yb in tr_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad(); loss = crit(mlp(xb), yb); loss.backward(); opt.step()
    mlp.eval()
    with torch.no_grad():
        vp = mlp(Xval_t).argmax(1).cpu().numpy()
    f1 = f1_score(yval, vp, average='macro')
    print(f'epoch {epoch:02d}  val F1-macro {f1:.4f}')
    if f1 > best_f1:
        best_f1 = f1; bad = 0
        best_state = {k: v.cpu().clone() for k, v in mlp.state_dict().items()}
    else:
        bad += 1
        if bad >= patience:
            print('early stop'); break
mlp.load_state_dict(best_state)
print('best val F1-macro:', round(best_f1, 4))

In [ ]:
mlp.eval()
with torch.no_grad():
    test_proba = F.softmax(mlp(torch.tensor(Xte).to(device)), 1).cpu().numpy()
    val_proba = F.softmax(mlp(torch.tensor(Xval).to(device)), 1).cpu().numpy()
ypred = test_proba.argmax(1)
acc = accuracy_score(yte, ypred)
f1m = f1_score(yte, ypred, average='macro')
print(f'TEST  Accuracy {acc:.4f}  |  F1-macro {f1m:.4f}  (baseline 0.6347)')
print(classification_report(yte, ypred, target_names=LABEL_NAMES, digits=4))

In [ ]:
# 예측확률/가중치 저장 (Late Fusion + 보고서용)
np.save(os.path.join(ART_DIR, 'mlp_test_proba.npy'), test_proba)
np.save(os.path.join(ART_DIR, 'mlp_val_proba.npy'), val_proba)
torch.save(mlp.state_dict(), os.path.join(ART_DIR, 'bert_mlp.pt'))
print('saved: mlp_test_proba / mlp_val_proba / bert_mlp.pt')

In [ ]:
# Confusion Matrix (라벨은 인덱스, 매핑은 아래 출력)
cm = confusion_matrix(yte, ypred)
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(N_CLASS)); ax.set_yticks(range(N_CLASS))
ax.set_xlabel('predicted'); ax.set_ylabel('true')
ax.set_title(f'BERT+MLP Confusion Matrix (F1-macro {f1m:.3f})')
for i in range(N_CLASS):
    for j in range(N_CLASS):
        ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=8)
plt.colorbar(im); plt.tight_layout()
plt.savefig(os.path.join(ART_DIR, 'confusion_matrix_bert_mlp.png'), dpi=150)
plt.show()
for i, n in enumerate(LABEL_NAMES):
    print(i, n)